# Treebank Token Comparison: XML vs CoNLL-U

Focuses on sentences where the two treebanks have **different token counts**, showing exactly which tokens were added or removed.

In [57]:
import xml.etree.ElementTree as ET
import re
import unicodedata
from collections import OrderedDict, Counter
from difflib import SequenceMatcher
from IPython.display import display, HTML

# ── Oxia → Tonos normalization ──────────────────────────────
# Greek treebanks mix oxia (U+1F71 ά) and tonos (U+03AC ά).
# NFC normalization maps oxia to tonos for all vowels.
def normalize(s):
    """NFC-normalize a string so oxia and tonos accent variants match."""
    return unicodedata.normalize('NFC', s) if s else s

# ── File paths ──────────────────────────────────────────────
# ── File paths ──────────────────────────────────────────────
XML_FILE = "/Users/gcrane/github/glaux-trees/public/xml/0086-034.xml"       # ← change to your XML file
CONLLU_FILE = "/Users/gcrane/github/conllu-viz/test-data/aristotle.poetics.tb.conllu" # ← change to your CoNLL-U file

In [58]:
def parse_xml(path):
    """Parse XML treebank → dict of {sentence_id: [{'form', 'lemma', 'postag', 'id'}]}"""
    tree = ET.parse(path)
    root = tree.getroot()
    sentences = OrderedDict()
    for sent in root.iter("sentence"):
        sid = sent.get("id")
        tokens = []
        for word in sent.findall("word"):
            tokens.append({
                "form": normalize(word.get("form")),
                "lemma": normalize(word.get("lemma")),
                "postag": word.get("postag", ""),
                "id": word.get("id", ""),
            })
        sentences[sid] = tokens
    return sentences


def parse_conllu(path):
    """Parse CoNLL-U file → dict of {sentence_id: [{'form', 'lemma', 'postag', 'id'}]}"""
    sentences = OrderedDict()
    current_id = None
    current_tokens = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("# sentence_id"):
                m = re.search(r"=\s*(.+)", line)
                if m:
                    current_id = m.group(1).strip()
            elif line == "":
                if current_id is not None and current_tokens:
                    sentences[current_id] = current_tokens
                current_tokens = []
            elif line.startswith("#"):
                continue
            else:
                cols = line.split("\t")
                if len(cols) >= 10:
                    tok_id = cols[0]
                    if "-" in tok_id or "." in tok_id:
                        continue
                    current_tokens.append({
                        "form": normalize(cols[1]),
                        "lemma": normalize(cols[2]),
                        "postag": cols[4],  # XPOS
                        "id": tok_id,
                    })
        if current_id is not None and current_tokens:
            sentences[current_id] = current_tokens

    return sentences


xml_sents = parse_xml(XML_FILE)
conllu_sents = parse_conllu(CONLLU_FILE)

print(f"XML sentences:    {len(xml_sents)}")
print(f"CoNLL-U sentences: {len(conllu_sents)}")

XML sentences:    598
CoNLL-U sentences: 598


In [59]:
# ── Find sentences with token-count mismatches ─────────────

all_ids = sorted(
    set(xml_sents.keys()) | set(conllu_sents.keys()),
    key=lambda x: int(x) if x.isdigit() else x,
)

count_mismatches = []
only_xml = []
only_conllu = []
ok = 0

for sid in all_ids:
    in_xml = sid in xml_sents
    in_conllu = sid in conllu_sents
    if in_xml and not in_conllu:
        only_xml.append(sid)
    elif in_conllu and not in_xml:
        only_conllu.append(sid)
    elif len(xml_sents[sid]) != len(conllu_sents[sid]):
        count_mismatches.append(sid)
    else:
        ok += 1

print(f"✅ {ok} sentences have the same token count")
print(f"❌ {len(count_mismatches)} sentences have different token counts")
if only_xml:
    print(f"⚠️  {len(only_xml)} sentences only in XML: {only_xml[:20]}")
if only_conllu:
    print(f"⚠️  {len(only_conllu)} sentences only in CoNLL-U: {only_conllu[:20]}")

✅ 585 sentences have the same token count
❌ 13 sentences have different token counts


In [60]:
# ── Diff display ──────────────────────────────────────────

def diff_tokens(sid):
    """
    Align XML and CoNLL-U tokens using SequenceMatcher on forms,
    then render an HTML table showing matched, inserted, and deleted tokens.
    """
    xml_toks = xml_sents.get(sid, [])
    conllu_toks = conllu_sents.get(sid, [])

    xml_forms = [t["form"] for t in xml_toks]
    conllu_forms = [t["form"] for t in conllu_toks]

    sm = SequenceMatcher(None, xml_forms, conllu_forms)
    opcodes = sm.get_opcodes()

    rows = []
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == "equal":
            for i, j in zip(range(i1, i2), range(j1, j2)):
                xt = xml_toks[i]
                ct = conllu_toks[j]
                rows.append(
                    f'<tr>'
                    f'<td>{xt["id"]}</td><td>{xt["form"]}</td><td>{xt["lemma"]}</td><td>{xt["postag"]}</td>'
                    f'<td style="text-align:center">＝</td>'
                    f'<td>{ct["id"]}</td><td>{ct["form"]}</td><td>{ct["lemma"]}</td><td>{ct["postag"]}</td>'
                    f'</tr>'
                )
        elif tag == "delete":
            for i in range(i1, i2):
                xt = xml_toks[i]
                rows.append(
                    f'<tr style="background:#fdd;">'
                    f'<td>{xt["id"]}</td><td><b>{xt["form"]}</b></td><td>{xt["lemma"]}</td><td>{xt["postag"]}</td>'
                    f'<td style="text-align:center; color:#c00;">← XML only</td>'
                    f'<td colspan="4"></td>'
                    f'</tr>'
                )
        elif tag == "insert":
            for j in range(j1, j2):
                ct = conllu_toks[j]
                rows.append(
                    f'<tr style="background:#dfd;">'
                    f'<td colspan="4"></td>'
                    f'<td style="text-align:center; color:#080;">CoNLL-U only →</td>'
                    f'<td>{ct["id"]}</td><td><b>{ct["form"]}</b></td><td>{ct["lemma"]}</td><td>{ct["postag"]}</td>'
                    f'</tr>'
                )
        elif tag == "replace":
            for i in range(i1, i2):
                xt = xml_toks[i]
                rows.append(
                    f'<tr style="background:#fdd;">'
                    f'<td>{xt["id"]}</td><td><b>{xt["form"]}</b></td><td>{xt["lemma"]}</td><td>{xt["postag"]}</td>'
                    f'<td style="text-align:center; color:#c00;">← XML only</td>'
                    f'<td colspan="4"></td>'
                    f'</tr>'
                )
            for j in range(j1, j2):
                ct = conllu_toks[j]
                rows.append(
                    f'<tr style="background:#dfd;">'
                    f'<td colspan="4"></td>'
                    f'<td style="text-align:center; color:#080;">CoNLL-U only →</td>'
                    f'<td>{ct["id"]}</td><td><b>{ct["form"]}</b></td><td>{ct["lemma"]}</td><td>{ct["postag"]}</td>'
                    f'</tr>'
                )

    html = f"""
    <h3>Sentence {sid} — XML: {len(xml_toks)} tokens, CoNLL-U: {len(conllu_toks)} tokens (Δ {len(conllu_toks) - len(xml_toks):+d})</h3>
    <table border="1" cellpadding="4" style="border-collapse:collapse; font-size:13px;">
    <tr style="background:#eee;">
      <th colspan="4">XML</th>
      <th></th>
      <th colspan="4">CoNLL-U</th>
    </tr>
    <tr style="background:#f5f5f5;">
      <th>id</th><th>form</th><th>lemma</th><th>postag</th>
      <th>status</th>
      <th>id</th><th>form</th><th>lemma</th><th>postag</th>
    </tr>
    {''.join(rows)}
    </table>
    """
    display(HTML(html))


print(f"{len(count_mismatches)} sentences to review")

13 sentences to review


In [61]:
# ── Show all mismatched sentences ─────────────────────────
# (If there are many, loop over a slice: count_mismatches[:20])

for sid in count_mismatches:
    if(int(sid)<550):
        continue
    diff_tokens(sid)

In [62]:
# ── Aggregate: what kinds of tokens are typically added/removed? ──

xml_only_forms = Counter()
conllu_only_forms = Counter()
xml_only_postags = Counter()
conllu_only_postags = Counter()

for sid in count_mismatches:
    xml_toks = xml_sents[sid]
    conllu_toks = conllu_sents[sid]
    xml_forms = [t["form"] for t in xml_toks]
    conllu_forms = [t["form"] for t in conllu_toks]

    sm = SequenceMatcher(None, xml_forms, conllu_forms)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag in ("delete", "replace"):
            for i in range(i1, i2):
                xml_only_forms[xml_toks[i]["form"]] += 1
                xml_only_postags[xml_toks[i]["postag"]] += 1
        if tag in ("insert", "replace"):
            for j in range(j1, j2):
                conllu_only_forms[conllu_toks[j]["form"]] += 1
                conllu_only_postags[conllu_toks[j]["postag"]] += 1

print("=== Tokens only in XML (most common) ===")
for form, n in xml_only_forms.most_common(20):
    print(f"  {form:20s}  ×{n}")

print("\n=== Tokens only in CoNLL-U (most common) ===")
for form, n in conllu_only_forms.most_common(20):
    print(f"  {form:20s}  ×{n}")

print("\n=== POS tags of XML-only tokens ===")
for tag, n in xml_only_postags.most_common(10):
    print(f"  {tag:15s}  ×{n}")

print("\n=== POS tags of CoNLL-U-only tokens ===")
for tag, n in conllu_only_postags.most_common(10):
    print(f"  {tag:15s}  ×{n}")

=== Tokens only in XML (most common) ===
  Καὶ                   ×2
  Τὰ                    ×2
  ,                     ×2
  Ἔστι                  ×1
  ἀλλὰ                  ×1
  διαιρέσει             ×1
  πλέων                 ×1

=== Tokens only in CoNLL-U (most common) ===
  "                     ×12
  ·                     ×7
  ,                     ×7
  τε                    ×5
  καὶ                   ×5
  δέ                    ×3
  μὲν                   ×3
  δὲ                    ×3
  τὸ                    ×3
  δίφρον                ×2
  καταθεὶς              ×2
  τράπεζαν              ×2
  .                     ×2
  δ᾽                    ×2
  πρὶν                  ×2
  πλέω                  ×2
  ἔστι                  ×1
  φαγέδαιναν            ×1
  ἥ                     ×1
  μου                   ×1

=== POS tags of XML-only tokens ===
  n-s---fd-        ×2
  u--------        ×2
  n-p---fn-        ×1
  a-p---fn-        ×1
  b--------        ×1
  n--------        ×1
  a-p---na-   

In [63]:
# ── Inspect a single sentence by ID ───────────────────────

for i in range(590,599):
    cursent = str(i)
    diff_tokens(cursent)  # ← change to any sentence id